In [1]:
import pandas as pd
import numpy as np
import numpy_financial as npf
import matplotlib.pyplot as plt
pd.set_option('display.max_rows', None)    # Show all rows
pd.set_option('display.max_columns', None)
pd.set_option("display.float_format", "{:,.2f}".format)

In [2]:
def load_data(filepath, ticker, start_date=None, end_date=None):
    price = pd.read_parquet(filepath)
    price_close = price[['Close']].dropna()
    price_close.columns = [ticker]
    
    if start_date:
        price_close = price_close[price_close.index >= pd.Timestamp(start_date)]
    if end_date:
        price_close = price_close[price_close.index <= pd.Timestamp(end_date)]

    monthly = price_close[ticker].groupby(pd.Grouper(freq="MS")).nth(0)

    return price_close, monthly

def calc_portfolio(price_close, monthly, investment):
    shares = investment / monthly
    cum_shares = shares.cumsum()
    daily_cum_shares = cum_shares.reindex(price_close.index, method="ffill")
    price_close['port_value'] = daily_cum_shares * price_close.iloc[:, 0]
    return price_close

def calc_metrics(price_close, monthly, investment):
    
    # ===== 基本數字 =====

    total_invested = [-investment] * len(monthly)
    final_value = total_invested.copy()
    final_value[-1] += price_close['port_value'].iloc[-1]
    #IRR
    monthly_irr = npf.irr(final_value)
    annualized_irr = (1 + monthly_irr) ** 12 - 1

    #CAGR 計算 (for lum sum usually, comment out for DCA)
    #years = (price_close.index[-1] - price_close.index[0]).days / 365 #find the total years
    #cagr = (price_close['port_value'].iloc[-1] / investment) ** (1 / years) - 1 
    #CAGR = (final value / initial value) ^ (1 / number of years)

    #Cash flow series
    cashflow_series = pd.Series(0.0, index=price_close.index)
    cashflow_series.loc[monthly.index] = investment
    price_close['strategy_return'] = (
        price_close['port_value'] - price_close['port_value'].shift(1) - cashflow_series
    ) / price_close['port_value'].shift(1)

    #Sharpe Ratio and Sortino Ratio
    ret = price_close['strategy_return'].dropna()
    mean_return = ret.mean()
    std_return = ret.std()

    volatility = std_return * (252 ** 0.5)
    sharpe = mean_return / std_return * (252 ** 0.5)
    downside = ret[ret < 0]
    sortino = mean_return / downside.std() * (252 ** 0.5)

    #MDD
    cum_max = price_close['port_value'].cummax()
    mdd = (price_close['port_value'] / cum_max - 1).min()

    return {
        "Total Invested": sum(total_invested),
        "Final Value":    final_value[-1],
        "IRR":            annualized_irr,
        #"CAGR":           cagr,
        "Volatility":     volatility,
        "Sharpe":         sharpe,
        "Sortino":        sortino,
        "MDD":            mdd,
    }

def run_backtest(ticker, start_date=None, end_date=None, investment=1000):
    filepath = f"data/{ticker}.parquet"
    price_close, monthly = load_data(filepath, ticker, start_date, end_date)
    price_close = calc_portfolio(price_close, monthly, investment)
    metrics = calc_metrics(price_close, monthly, investment)
    return metrics



In [3]:
tickers = ["SPY", "QQQ", "IWY"]
start = "2022-01-01"
end = "2025-01-01"
results = {ticker: run_backtest(ticker, start, end) for ticker in tickers}
pd.DataFrame(results)

,SPY,QQQ,IWY
Total Invested,"-36,000.00","-36,000.00","-36,000.00"
Final Value,"47,018.95","50,816.58","53,066.05"
IRR,0.21,0.27,0.30
Volatility,0.18,0.24,0.22
Sharpe,0.56,0.49,0.58
Sortino,0.80,0.71,0.84
MDD,-0.13,-0.15,-0.14
